# Customer Behavior Analysis Using Online Retail Data

## Introduction

This project explores customer purchasing behavior using online retail transaction data. The goal is to better understand how customers interact with products by analyzing both what items are purchased together and how purchasing patterns evolve over time.

By applying data mining techniques, this project aims to uncover insights that go beyond simple co-purchase relationships and provide a deeper understanding of real customer behavior.

## Research Question

The primary goal of this project is to answer the following questions:

- What products are frequently purchased together?
- How do customer purchasing patterns evolve over time?
- What additional insights can sequential pattern mining provide that traditional association rule mining may miss?

## Data Overview

The dataset used in this project is the Online Retail dataset, which contains transactional data from a UK-based online retailer.

Key characteristics of the dataset include:
- Approximately 3,800+ unique products
- Over 18,000 invoices
- An average basket size of about 21 items per transaction
- The majority of transactions (around 89%) originate from the United Kingdom

This dataset provides a strong foundation for analyzing customer purchasing behavior at scale.

## Data Preprocessing

Before applying any data mining techniques, the dataset was cleaned and transformed to ensure meaningful analysis.

Key preprocessing steps included:
- Removing missing or invalid values
- Filtering out canceled transactions
- Converting transaction data into a format suitable for analysis
- Creating a transaction matrix for frequent itemset mining

These steps ensured that the data was structured properly for both association rule mining and sequential pattern analysis.

In [3]:
from google.colab import files
uploaded = files.upload()

Saving online+retail.zip to online+retail (1).zip


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_excel("Online Retail.xlsx")

# Clean column names
df.columns = df.columns.str.strip()

# Convert date and numeric columns
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")

# Remove canceled transactions
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]

# Remove invalid purchases
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

# Remove missing key values
df = df.dropna(subset=["InvoiceNo", "StockCode", "Description", "CustomerID"])

# Clean product descriptions
df["Description"] = df["Description"].astype(str).str.strip()

# Quick check
print("Cleaned dataset shape:", df.shape)
print("Unique products:", df["Description"].nunique())
print("Unique invoices:", df["InvoiceNo"].nunique())
print("Average basket size:", df.groupby("InvoiceNo")["StockCode"].count().mean())
print("Top country percentage:", (df["Country"].value_counts().iloc[0] / len(df)) * 100)

Cleaned dataset shape: (397884, 8)
Unique products: 3866
Unique invoices: 18532
Average basket size: 21.470105763004533
Top country percentage: 89.05133154386705


After cleaning the dataset, I removed canceled transactions, invalid purchase quantities, missing customer IDs, and missing product descriptions. This helped ensure that the final dataset only included valid customer purchases that could be used for both frequent itemset mining and sequential pattern analysis.

## Methodology

Two primary techniques were used in this project to analyze customer behavior:

### Frequent Itemset Mining (Apriori)
The Apriori algorithm was used to identify products that are frequently purchased together. This method helps uncover associations between items based on support, confidence, and lift metrics.

### Sequential Pattern Mining
Sequential pattern mining was applied to identify the order in which products are purchased over time. This approach provides deeper insight into customer behavior by capturing temporal relationships that traditional methods cannot detect.

In [6]:
# Install mlxtend if needed
!pip install mlxtend

In [7]:
from mlxtend.frequent_patterns import apriori, association_rules

# Create basket format for Apriori
basket = df.groupby(["InvoiceNo", "Description"])["Quantity"] \
           .sum() \
           .unstack() \
           .fillna(0)

# Convert quantities into 1/0 values
basket = basket.map(lambda x: 1 if x > 0 else 0)

basket.head()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Description,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 DAISY PEGS IN WOOD BOX,12 EGG HOUSE PAINTED WOOD,12 HANGING EGGS HAND PAINTED,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,12 PENCILS SMALL TUBE RED RETROSPOT,12 PENCILS SMALL TUBE SKULL,...,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536365,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536366,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536367,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536368,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536369,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [8]:
# Run Apriori frequent itemset mining
frequent_itemsets = apriori(
    basket,
    min_support=0.02,
    use_colnames=True
)

frequent_itemsets = frequent_itemsets.sort_values(
    by="support",
    ascending=False
)

frequent_itemsets.head(10)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,support,itemsets
192,0.106357,(WHITE HANGING HEART T-LIGHT HOLDER)
154,0.091895,(REGENCY CAKESTAND 3 TIER)
81,0.086337,(JUMBO BAG RED RETROSPOT)
124,0.074412,(PARTY BUNTING)
11,0.074196,(ASSORTED COLOUR BIRD ORNAMENT)
102,0.069501,(LUNCH BAG RED RETROSPOT)
166,0.061839,(SET OF 3 CAKE TINS PANTRY DESIGN)
139,0.059303,(POSTAGE)
95,0.056767,(LUNCH BAG BLACK SKULL.)
116,0.055526,(PACK OF 72 RETROSPOT CAKE CASES)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


The table above shows the most frequently purchased individual products, indicating which items are most popular among customers.

In [9]:
# Generate association rules
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.0
)

rules = rules.sort_values(
    by=["lift", "confidence"],
    ascending=False
)

rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,antecedents,consequents,support,confidence,lift
63,(PINK REGENCY TEACUP AND SAUCER),"(GREEN REGENCY TEACUP AND SAUCER, ROSES REGENC...",0.021045,0.701439,24.027846
62,"(GREEN REGENCY TEACUP AND SAUCER, ROSES REGENC...",(PINK REGENCY TEACUP AND SAUCER),0.021045,0.720887,24.027846
61,"(PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY...",(GREEN REGENCY TEACUP AND SAUCER),0.021045,0.894495,23.989564
64,(GREEN REGENCY TEACUP AND SAUCER),"(PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY...",0.021045,0.564399,23.989564
18,(PINK REGENCY TEACUP AND SAUCER),(GREEN REGENCY TEACUP AND SAUCER),0.024822,0.827338,22.188466
19,(GREEN REGENCY TEACUP AND SAUCER),(PINK REGENCY TEACUP AND SAUCER),0.024822,0.665702,22.188466
60,"(GREEN REGENCY TEACUP AND SAUCER, PINK REGENCY...",(ROSES REGENCY TEACUP AND SAUCER),0.021045,0.847826,20.066300
65,(ROSES REGENCY TEACUP AND SAUCER),"(GREEN REGENCY TEACUP AND SAUCER, PINK REGENCY...",0.021045,0.498084,20.066300
28,(PINK REGENCY TEACUP AND SAUCER),(ROSES REGENCY TEACUP AND SAUCER),0.023527,0.784173,18.559754
29,(ROSES REGENCY TEACUP AND SAUCER),(PINK REGENCY TEACUP AND SAUCER),0.023527,0.556833,18.559754


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

These rules highlight strong relationships between products, showing which items are likely to be purchased together within the same transaction.

In [10]:
# Prepare customer-level sequences for sequential pattern analysis
df_sorted = df.sort_values(["CustomerID", "InvoiceDate"])

customer_sequences = df_sorted.groupby("CustomerID")["Description"].apply(list)

customer_sequences.head()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,Description
CustomerID,
12346.0,[MEDIUM CERAMIC TOP STORAGE JAR]
12347.0,"[BLACK CANDELABRA T-LIGHT HOLDER, AIRLINE BAG ..."
12348.0,"[72 SWEETHEART FAIRY CAKE CASES, 60 CAKE CASES..."
12349.0,"[PARISIENNE CURIO CABINET, SWEETHEART WALL TID..."
12350.0,"[CHOCOLATE THIS WAY METAL SIGN, METAL SIGN NEI..."


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


These sequences show how customers move from one purchase to another over time, revealing patterns that are not visible through traditional association analysis.

In [11]:
from collections import Counter
from itertools import pairwise

# Count common two-item purchase sequences
sequence_counts = Counter()

for sequence in customer_sequences:
    cleaned_sequence = [item for item in sequence if pd.notna(item)]

    for first_item, second_item in pairwise(cleaned_sequence):
        if first_item != second_item:
            sequence_counts[(first_item, second_item)] += 1

top_sequences = pd.DataFrame(
    sequence_counts.most_common(10),
    columns=["Sequence", "Count"]
)

top_sequences

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,Sequence,Count
0,"(GARDENERS KNEELING PAD CUP OF TEA, GARDENERS ...",218
1,"(WOODEN FRAME ANTIQUE WHITE, WOODEN PICTURE FR...",195
2,"(GARDENERS KNEELING PAD KEEP CALM, GARDENERS K...",185
3,"(PINK REGENCY TEACUP AND SAUCER, GREEN REGENCY...",183
4,"(ALARM CLOCK BAKELIKE RED, ALARM CLOCK BAKELIK...",170
5,"(ROSES REGENCY TEACUP AND SAUCER, GREEN REGENC...",167
6,"(PAPER CHAIN KIT 50'S CHRISTMAS, PAPER CHAIN K...",167
7,"(GREEN REGENCY TEACUP AND SAUCER, PINK REGENCY...",164
8,"(ALARM CLOCK BAKELIKE GREEN, ALARM CLOCK BAKEL...",162
9,"(WOODEN PICTURE FRAME WHITE FINISH, WOODEN FRA...",161


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

This section applies two different approaches to understand customer behavior. Apriori identifies products that frequently appear together in the same transaction, while the sequential pattern analysis looks at the order of purchases across customer histories. Using both methods helps compare simple co-purchase relationships with time-based purchasing behavior.

## Results

The Apriori algorithm identified several high-frequency products that appear consistently across transactions. Items such as *WHITE HANGING HEART T-LIGHT HOLDER*, *REGENCY CAKESTAND 3 TIER*, and *JUMBO BAG RED RETROSPOT* had the highest support values, indicating they are among the most commonly purchased items.

The association rules revealed strong relationships between products, particularly within themed product groups. For example, variations of *REGENCY TEACUP AND SAUCER* items frequently appeared together, with high confidence and lift values exceeding 20. This suggests a strong likelihood that customers purchasing one variation of a product are likely to purchase related variations as well.

Sequential pattern analysis provided additional insight into purchasing behavior over time. The most common sequences included transitions between related household or decorative items, such as gardening products and kitchenware. These patterns highlight that customers tend to make follow-up purchases within similar product categories.

Overall, the results show that while some products are consistently popular, purchasing behavior is also influenced by product relationships and sequential buying patterns.